# E6-medvae：vae 后验初始化 + BaM 100 步 refinement（cifar10，MedVAE 骨干）

协议同论文 §7.5 / MLP 版 e6-iter notebook：从收敛模型的 encoder 后验 (mu, diag(exp(logvar))) 初始化，BaM refinement T=100、B=300、λ=7500、σ²=1.0、v2。
与 MLP 版的差异：骨干换成 E5 的 MedVAE（AutoencoderKL，隐空间 4×16×16 展平 D=1024）；直接加载 E5 训好的 checkpoint，不重新训练。

⚠️ D=1024 时 BaM 每步含 1024×1024 矩阵开方（CPU scipy sqrtm），**每张图约数分钟量级**，因此默认从 test set 固定种子抽 N_IMAGES=100 张（vae/sdvae 两本同种子同抽样，保证逐图配对）。跑第 1 张图后看打印的 s/img，再决定是否调小 N_IMAGES 或分片。


In [1]:
import os
import math
import time
import random
import numpy as np
from PIL import Image

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange
import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)  # D=1024 时 mat=I+4UV 条件数 ~(λ·score尺度)²，f32 开方/求逆撑不住（实测非 PSD 病态）；f64 每步约慢 30-50%


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


SEED = 42
set_seed(SEED)
MASTER_JAX_KEY = jax.random.PRNGKey(SEED)


## config

In [2]:
# ===== E6-medvae refinement 配置（唯一需要按方法/分片改动的 cell）=====
METHOD_TAG = "vae"   # sdvae / vae
DATASET = "cifar10"

# ⚠️ 路径按 results_medvae_0706 目录名推测，跑前务必 ls 确认
CKPT = "/home/yzm/ipy/checkpoints/medvae-re-cifar10-06-22_14-53/vae_kl_1e-05/medvae_vae_kl_1e-05_best.pth"

N_IMAGES = 100      # 从 test set 固定种子抽样的图像数（None=全测试集；D=1024 全集不可行）
SHARD_ID = 0        # 本分片编号 0..NUM_SHARDS-1；并行时复制本 notebook 或转 .py，各改此值
NUM_SHARDS = 1      # 总分片数

T = 100             # refinement 迭代步数（论文 §7.5 协议）
B = 300             # BaM 每步采样数
LAMBDA_T = 7500     # ⚠️ D=1024 实测 λ=7500 触发非 PSD 病态（B=300/1024 皆报）；先跑下方 λ preflight 小格子选值再改这里跑全量
SIGMA2 = 1.0        # 与训练时 sd 的 sigma2 一致（数据在 [-1,1]）
BAM_VERSION = "v2"
JITTER = 1e-6
SEED = 42

N_TAG = f"N{N_IMAGES}" if N_IMAGES is not None else "Nall"

cuda_num = 2
SAVE_DIR = "/home/yzm/ipy/e6_iter_medvae_results"
os.makedirs(SAVE_DIR, exist_ok=True)


class Config:
    DATASET_NAME = "cifar10"
    DATASET_DIR = "/home/yzm/data_cifar10"
    TRAIN_DATASET_DIR = os.path.join(DATASET_DIR, "train")
    TEST_DATASET_DIR = os.path.join(DATASET_DIR, "test")
    IMAGE_KEY = "img"
    IMG_SIZE = 32
    IMG_CHANNELS = 3
    CUDA_NUM = cuda_num


config = Config()
device = torch.device(f"cuda:{config.CUDA_NUM}" if torch.cuda.is_available() else "cpu")
print("device:", device)
print(f"{DATASET} / {METHOD_TAG} / {N_TAG} / shard {SHARD_ID}/{NUM_SHARDS}")


device: cuda:2
cifar10 / vae / N100 / shard 0/1


## MedVAE 骨干（与 E5 训练 notebook 逐字一致，ddconfig 不许改）

In [3]:
# ==========================================
# MedVAE 核心组件: Encoder & Decoder
# ==========================================

def nonlinearity(x):
    """Swish激活函数"""
    return x * torch.sigmoid(x)

def Normalize(in_channels, num_groups=32):
    return nn.GroupNorm(num_groups=num_groups, num_channels=in_channels, eps=1e-6, affine=True)

class LinearAttention(nn.Module):
    def __init__(self, dim, heads=4, dim_head=32):
        super().__init__()
        self.heads = heads
        hidden_dim = dim_head * heads
        self.to_qkv = nn.Conv2d(dim, hidden_dim * 3, 1, bias=False)
        self.to_out = nn.Conv2d(hidden_dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.to_qkv(x)
        q, k, v = rearrange(qkv, "b (qkv heads c) h w -> qkv b heads c (h w)", heads=self.heads, qkv=3)
        k = k.softmax(dim=-1)
        context = torch.einsum("bhdn,bhen->bhde", k, v)
        out = torch.einsum("bhde,bhdn->bhen", context, q)
        out = rearrange(out, "b heads c (h w) -> b (heads c) h w", heads=self.heads, h=h, w=w)
        return self.to_out(out)

class LinAttnBlock(LinearAttention):
    def __init__(self, in_channels):
        super().__init__(dim=in_channels, heads=1, dim_head=in_channels)

class AttnBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.in_channels = in_channels
        self.norm = Normalize(in_channels)
        self.q = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.k = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.v = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        self.proj_out = nn.Conv2d(in_channels, in_channels, kernel_size=1)

    def forward(self, x):
        h_ = self.norm(x)
        q, k, v = self.q(h_), self.k(h_), self.v(h_)
        b, c, h, w = q.shape
        q = q.reshape(b, c, h * w).permute(0, 2, 1)
        k = k.reshape(b, c, h * w)
        w_ = torch.bmm(q, k) * (int(c) ** (-0.5))
        w_ = F.softmax(w_, dim=2)
        v = v.reshape(b, c, h * w)
        w_ = w_.permute(0, 2, 1)
        h_ = torch.bmm(v, w_).reshape(b, c, h, w)
        return x + self.proj_out(h_)

def make_attn(in_channels, attn_type="vanilla"):
    if attn_type == "vanilla":
        return AttnBlock(in_channels)
    elif attn_type == "none":
        return nn.Identity(in_channels)
    else:
        return LinAttnBlock(in_channels)

class Upsample(nn.Module):
    def __init__(self, in_channels, with_conv):
        super().__init__()
        self.with_conv = with_conv
        if self.with_conv:
            self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if self.with_conv:
            x = self.conv(x)
        return x

class Downsample(nn.Module):
    def __init__(self, in_channels, with_conv):
        super().__init__()
        self.with_conv = with_conv
        if self.with_conv:
            self.conv = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=2, padding=0)

    def forward(self, x):
        if self.with_conv:
            x = F.pad(x, (0, 1, 0, 1), mode="constant", value=0)
            x = self.conv(x)
        else:
            x = F.avg_pool2d(x, kernel_size=2, stride=2)
        return x

class ResnetBlock(nn.Module):
    def __init__(self, *, in_channels, out_channels=None, conv_shortcut=False, dropout, temb_channels=512):
        super().__init__()
        self.in_channels = in_channels
        out_channels = in_channels if out_channels is None else out_channels
        self.out_channels = out_channels
        self.use_conv_shortcut = conv_shortcut

        self.norm1 = Normalize(in_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        if temb_channels > 0:
            self.temb_proj = nn.Linear(temb_channels, out_channels)
        self.norm2 = Normalize(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        
        if self.in_channels != self.out_channels:
            if self.use_conv_shortcut:
                self.conv_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
            else:
                self.nin_shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x, temb):
        h = nonlinearity(self.norm1(x))
        h = self.conv1(h)
        if temb is not None:
            h = h + self.temb_proj(nonlinearity(temb))[:, :, None, None]
        h = nonlinearity(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)
        if self.in_channels != self.out_channels:
            x = self.conv_shortcut(x) if self.use_conv_shortcut else self.nin_shortcut(x)
        return x + h

In [4]:
# ==========================================
# Encoder 和 Decoder 类
# ==========================================

class Encoder(nn.Module):
    def __init__(self, *, ch, out_ch, ch_mult=(1, 2, 4, 8), num_res_blocks, attn_resolutions,
                 dropout=0.0, resamp_with_conv=True, in_channels, resolution, z_channels,
                 double_z=True, use_linear_attn=False, attn_type="vanilla", **ignore_kwargs):
        super().__init__()
        if use_linear_attn:
            attn_type = "linear"
        self.ch = ch
        self.temb_ch = 0
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution
        self.in_channels = in_channels

        # downsampling
        self.conv_in = nn.Conv2d(in_channels, self.ch, kernel_size=3, stride=1, padding=1)
        curr_res = resolution
        in_ch_mult = (1,) + tuple(ch_mult)
        self.down = nn.ModuleList()
        
        for i_level in range(self.num_resolutions):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_in = ch * in_ch_mult[i_level]
            block_out = ch * ch_mult[i_level]
            for i_block in range(self.num_res_blocks):
                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, temb_channels=self.temb_ch, dropout=dropout))
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(make_attn(block_in, attn_type=attn_type))
            down = nn.Module()
            down.block = block
            down.attn = attn
            if i_level != self.num_resolutions - 1:
                down.downsample = Downsample(block_in, resamp_with_conv)
                curr_res = curr_res // 2
            self.down.append(down)

        # middle
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, temb_channels=self.temb_ch, dropout=dropout)
        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, temb_channels=self.temb_ch, dropout=dropout)

        # end
        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, 2 * z_channels if double_z else z_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        temb = None
        hs = [self.conv_in(x)]
        for i_level in range(self.num_resolutions):
            for i_block in range(self.num_res_blocks):
                h = self.down[i_level].block[i_block](hs[-1], temb)
                if len(self.down[i_level].attn) > 0:
                    h = self.down[i_level].attn[i_block](h)
                hs.append(h)
            if i_level != self.num_resolutions - 1:
                hs.append(self.down[i_level].downsample(hs[-1]))
        h = hs[-1]
        h = self.mid.block_1(h, temb)
        h = self.mid.attn_1(h)
        h = self.mid.block_2(h, temb)
        h = self.norm_out(h)
        h = nonlinearity(h)
        h = self.conv_out(h)
        return h


class Decoder(nn.Module):
    def __init__(self, *, ch, out_ch, ch_mult=(1, 2, 4, 8), num_res_blocks, attn_resolutions,
                 dropout=0.0, resamp_with_conv=True, in_channels, resolution, z_channels,
                 give_pre_end=False, tanh_out=False, use_linear_attn=False, attn_type="vanilla", **ignorekwargs):
        super().__init__()
        if use_linear_attn:
            attn_type = "linear"
        self.ch = ch
        self.temb_ch = 0
        self.num_resolutions = len(ch_mult)
        self.num_res_blocks = num_res_blocks
        self.resolution = resolution
        self.in_channels = in_channels
        self.give_pre_end = give_pre_end
        self.tanh_out = tanh_out

        block_in = ch * ch_mult[self.num_resolutions - 1]
        curr_res = resolution // 2 ** (self.num_resolutions - 1)

        # z to block_in
        self.conv_in = nn.Conv2d(z_channels, block_in, kernel_size=3, stride=1, padding=1)

        # middle
        self.mid = nn.Module()
        self.mid.block_1 = ResnetBlock(in_channels=block_in, out_channels=block_in, temb_channels=self.temb_ch, dropout=dropout)
        self.mid.attn_1 = make_attn(block_in, attn_type=attn_type)
        self.mid.block_2 = ResnetBlock(in_channels=block_in, out_channels=block_in, temb_channels=self.temb_ch, dropout=dropout)

        # upsampling
        self.up = nn.ModuleList()
        for i_level in reversed(range(self.num_resolutions)):
            block = nn.ModuleList()
            attn = nn.ModuleList()
            block_out = ch * ch_mult[i_level]
            for i_block in range(self.num_res_blocks + 1):
                block.append(ResnetBlock(in_channels=block_in, out_channels=block_out, temb_channels=self.temb_ch, dropout=dropout))
                block_in = block_out
                if curr_res in attn_resolutions:
                    attn.append(make_attn(block_in, attn_type=attn_type))
            up = nn.Module()
            up.block = block
            up.attn = attn
            if i_level != 0:
                up.upsample = Upsample(block_in, resamp_with_conv)
                curr_res = curr_res * 2
            self.up.insert(0, up)

        # end
        self.norm_out = Normalize(block_in)
        self.conv_out = nn.Conv2d(block_in, out_ch, kernel_size=3, stride=1, padding=1)

    def forward(self, z):
        temb = None
        h = self.conv_in(z)
        h = self.mid.block_1(h, temb)
        h = self.mid.attn_1(h)
        h = self.mid.block_2(h, temb)
        for i_level in reversed(range(self.num_resolutions)):
            for i_block in range(self.num_res_blocks + 1):
                h = self.up[i_level].block[i_block](h, temb)
                if len(self.up[i_level].attn) > 0:
                    h = self.up[i_level].attn[i_block](h)
            if i_level != 0:
                h = self.up[i_level].upsample(h)
        if self.give_pre_end:
            return h
        h = self.norm_out(h)
        h = nonlinearity(h)
        h = self.conv_out(h)
        if self.tanh_out:
            h = torch.tanh(h)
        return h

In [5]:
# ==========================================
# DiagonalGaussianDistribution (潜在空间分布)
# ==========================================

class DiagonalGaussianDistribution:
    def __init__(self, parameters, deterministic=False):
        self.parameters = parameters
        self.mean, self.logvar = torch.chunk(parameters, 2, dim=1)
        self.logvar = torch.clamp(self.logvar, -30.0, 20.0)
        self.deterministic = deterministic
        self.std = torch.exp(0.5 * self.logvar)
        self.var = torch.exp(self.logvar)
        if self.deterministic:
            self.var = self.std = torch.zeros_like(self.mean).to(device=self.parameters.device)

    def sample(self):
        x = self.mean + self.std * torch.randn(self.mean.shape).to(device=self.parameters.device)
        return x

    def kl(self, other=None):
        if self.deterministic:
            return torch.Tensor([0.0])
        if other is None:
            return 0.5 * torch.sum(torch.pow(self.mean, 2) + self.var - 1.0 - self.logvar, dim=[1, 2, 3])
        else:
            return 0.5 * torch.sum(
                torch.pow(self.mean - other.mean, 2) / other.var + self.var / other.var - 1.0 - self.logvar + other.logvar,
                dim=[1, 2, 3],
            )

    def nll(self, sample, dims=[1, 2, 3]):
        if self.deterministic:
            return torch.Tensor([0.0])
        logtwopi = np.log(2.0 * np.pi)
        return 0.5 * torch.sum(logtwopi + self.logvar + torch.pow(sample - self.mean, 2) / self.var, dim=dims)

    def mode(self):
        return self.mean

In [6]:
# ==========================================
# AutoencoderKL 模型 (更适合 CIFAR10 重建的配置)
# ==========================================

class AutoencoderKL(nn.Module):
    """
    2D VAE 模型
    - 4x 空间压缩
    - RGB 三通道输入
    - 更大的潜在容量用于提升 CIFAR10 重建质量
    """
    def __init__(self, ddconfig, embed_dim, ckpt_path=None, ignore_keys=[], apply_channel_ds=False):
        super().__init__()
        self.encoder = Encoder(**ddconfig)
        self.decoder = Decoder(**ddconfig)

        assert ddconfig["double_z"]
        self.quant_conv = nn.Conv2d(2 * ddconfig["z_channels"], 2 * embed_dim, 1)
        self.post_quant_conv = nn.Conv2d(embed_dim, ddconfig["z_channels"], 1)
        self.embed_dim = embed_dim
        self.latent_downsample_factor = 2 ** (len(ddconfig["ch_mult"]) - 1)
        self.latent_resolution = ddconfig["resolution"] // self.latent_downsample_factor

        self.apply_channel_ds = apply_channel_ds
        if self.apply_channel_ds:
            self.channel_ds = nn.Sequential(
                nn.Conv2d(self.embed_dim, 64, 1),
                nn.ReLU(),
                nn.Conv2d(64, 64, 3, padding="same"),
                nn.ReLU(),
                nn.Conv2d(64, self.embed_dim, 1),
            )
            self.channel_proj = nn.Conv2d(self.embed_dim, self.embed_dim, 1)

        if ckpt_path is not None:
            self.init_from_ckpt(ckpt_path, ignore_keys=ignore_keys)

    def init_from_ckpt(self, path, ignore_keys=[]):
        sd = torch.load(path, map_location="cpu")
        if "state_dict" in sd:
            sd = sd["state_dict"]
        keys = list(sd.keys())
        for k in keys:
            for ik in ignore_keys:
                if k.startswith(ik):
                    print(f"Deleting key {k} from state_dict.")
                    del sd[k]
        self.load_state_dict(sd, strict=False)
        print(f"Restored from {path}")

    def encode(self, x):
        """编码: 图像 -> 潜在分布"""
        h = self.encoder(x)
        moments = self.quant_conv(h)
        posterior = DiagonalGaussianDistribution(moments)
        return posterior

    def decode(self, z):
        """解码: 潜在向量 -> 图像"""
        z = self.post_quant_conv(z)
        dec = self.decoder(z)
        return dec

    def forward(self, input, sample_posterior=True):
        """前向传播"""
        posterior = self.encode(input)
        if sample_posterior:
            z = posterior.sample()
        else:
            z = posterior.mode()

        if self.apply_channel_ds:
            z = self.channel_proj(self.channel_ds(z) + z)

        dec = self.decode(z)
        return dec, posterior, z

    def get_last_layer(self):
        """获取decoder最后一层权重 (用于自适应权重计算)"""
        return self.decoder.conv_out.weight


ddconfig = {
    "double_z": True,
    "z_channels": 4,
    "resolution": config.IMG_SIZE,
    "in_channels": config.IMG_CHANNELS,
    "out_ch": config.IMG_CHANNELS,
    "ch": 32,
    "ch_mult": [1, 2],
    "num_res_blocks": 2,
    "attn_resolutions": [config.IMG_SIZE // 2],  # 与基准 cifar10 (32//2=16) 一致，按分辨率适配
    "dropout": 0.0
}
embed_dim = 4
latent_downsample_factor = 2 ** (len(ddconfig["ch_mult"]) - 1)
latent_resolution = ddconfig["resolution"] // latent_downsample_factor

print(f"模型配置: medvae_{config.DATASET_NAME}_recon")
print(f"  输入分辨率: {ddconfig['resolution']}x{ddconfig['resolution']}")
print(f"  输入通道: {ddconfig['in_channels']}")
print(f"  潜在空间: {embed_dim} 通道, {latent_resolution}x{latent_resolution} 分辨率")
print(f"  空间下采样倍数: {latent_downsample_factor}x")

模型配置: medvae_cifar10_recon
  输入分辨率: 32x32
  输入通道: 3
  潜在空间: 4 通道, 16x16 分辨率
  空间下采样倍数: 2x


## data (test only)

In [7]:
# ==========================================
# cifar10 Hugging Face 数据集 (VAE训练用，归一化到[-1,1])
# ==========================================
class HFImageDataset(Dataset):
    """
    Hugging Face cifar10 数据集封装
    输出图像归一化到 [-1, 1] 范围
    """
    def __init__(self, split='train', augment=False):
        split_to_path = {
            'train': config.TRAIN_DATASET_DIR,
            'test': config.TEST_DATASET_DIR,
        }
        if split not in split_to_path:
            raise ValueError(f"Unsupported split: {split}")

        self.split = split
        self.dataset_path = split_to_path[split]
        self.image_key = config.IMAGE_KEY
        self.dataset = datasets.load_from_disk(self.dataset_path)
        transform_ops = [
            transforms.Resize(config.IMG_SIZE),
            transforms.CenterCrop(config.IMG_SIZE),
        ]
        if augment:
            transform_ops.append(transforms.RandomHorizontalFlip())
        transform_ops.extend([
            transforms.ToTensor(),
            transforms.Normalize([0.5] * config.IMG_CHANNELS, [0.5] * config.IMG_CHANNELS),
        ])
        self.transform = transforms.Compose(transform_ops)

        print(f"[{self.split}] 数据集路径: {self.dataset_path}")
        print(f"[{self.split}] 数据集大小: {len(self.dataset)}")
        print(f"[{self.split}] 数据集列: {self.dataset.column_names}")

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img = self.dataset[idx][self.image_key]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert("RGB" if config.IMG_CHANNELS == 3 else "L")
        return self.transform(img)


In [8]:
# E6 只用 test set；单张取 test_dataset[i] -> (C,H,W)，值域 [-1,1]
test_dataset = HFImageDataset(split='test', augment=False)
print("test:", len(test_dataset), "images")


[test] 数据集路径: /home/yzm/data_cifar10/test
[test] 数据集大小: 10000
[test] 数据集列: ['img', 'label']
test: 10000 images


## 加载 E5 checkpoint（不训练）

In [9]:
model = AutoencoderKL(ddconfig, embed_dim, apply_channel_ds=False).to(device)
ck = torch.load(CKPT, map_location=device, weights_only=False)
state = ck["model_state_dict"] if "model_state_dict" in ck else ck
model.load_state_dict(state, strict=True)
model.eval()
for p_ in model.parameters():
    p_.requires_grad_(False)

_expected_mode = {"sdvae": "sd", "vae": "vae"}[METHOD_TAG]
_ck_mode = ck.get("loss_mode", None)
print(f"loaded: {CKPT}")
print(f"  epoch={ck.get('epoch', '?')}  val_loss={ck.get('val_loss', '?')}  loss_mode={_ck_mode}")
assert _ck_mode == _expected_mode, (
    f"checkpoint loss_mode={_ck_mode} 与 METHOD_TAG={METHOD_TAG}(期望 {_expected_mode}) 不符，请核对 CKPT 路径")

EMBED_DIM = model.embed_dim
LATENT_RES = model.latent_resolution
LATENT_DIM = EMBED_DIM * LATENT_RES * LATENT_RES
print(f"latent: {EMBED_DIM}x{LATENT_RES}x{LATENT_RES} -> D={LATENT_DIM}")

# sanity: 前 8 张 test 图的 amortized 后验均值重构 SSE（[0,1] 显示标度），确认加载的是训好的模型
with torch.no_grad():
    _xs = torch.stack([test_dataset[i] for i in range(8)]).to(device)
    _post = model.encode(_xs)
    _rec = model.decode(_post.mode())
    _sse = (((_rec - _xs) / 2) ** 2).flatten(1).sum(1)
print("sanity SSE (前8张, 0-1标度):", [f"{v:.2f}" for v in _sse.cpu().tolist()])


loaded: /home/yzm/ipy/checkpoints/medvae-re-cifar10-06-22_14-53/vae_kl_1e-05/medvae_vae_kl_1e-05_best.pth
  epoch=9  val_loss=0.04977867016686669  loss_mode=vae
latent: 4x16x16 -> D=1024
sanity SSE (前8张, 0-1标度): ['2.87', '1.33', '1.06', '2.27', '3.52', '1.68', '3.70', '1.02']


## BaM（与 MLP 版 e6-iter notebook 逐字一致）

In [10]:
import jax
import jax.numpy as jnp
from jax import jit, random
from jax.scipy.linalg import sqrtm as sqrtm_jsp
from scipy.linalg import sqrtm as sqrtm_sp
import numpy as np
import scipy.sparse as spys
from jax.lib import xla_bridge
import jax.debug

class BaM_V0:
    """
    Wrapper class for using BaM updates to fit a distribution
    """
    def __init__(self, D, lp, lp_g, use_lowrank=False, jit_compile=True):
        """
        Inputs:
          D: (int) Dimensionality (number) of parameters
          lp : Function to evaluate target log-probability distribution.
               (Only used in monitor, not for fitting)
          lp_g : Function to evaluate score, i.e. the gradient of the target log-probability distribution
        """
        #print("Using BaM V0")
        self.D = D
        self.lp = lp
        self.lp_g = lp_g
        self.use_lowrank = use_lowrank
        if use_lowrank:
            print("Using lowrank update")
        self.jit_compile = jit_compile
        if not jit_compile:
            print("Not using jit compilation. This may take longer than it needs to.")

    def compute_Q_host(U_B):
        U, B = U_B
        UU, DD, VV = spys.linalg.svds(U, k=B)
        return UU * np.sqrt(DD)

    def compute_Q(self, U_B):
        result_shape = jax.ShapeDtypeStruct((U_B[0].shape[0], U_B[1]), U_B[0].dtype)
        return jax.pure_callback(self.compute_Q_host, result_shape, U_B)

    def get_sqrt(self, M):
        if xla_bridge.get_backend().platform == 'gpu':
            result_shape = jax.ShapeDtypeStruct(M.shape, M.dtype)
            M_root = jax.pure_callback(lambda x:sqrtm_sp(x).astype(M.dtype), result_shape, M) # sqrt can be complex sometimes, we only want real part
        elif xla_bridge.get_backend().platform == 'cpu':
            M_root = sqrtm_jsp(M)
        else:
            print("Backend not recongnized in get_sqrt function. Should be either gpu or cpu")
            raise
        return M_root.real


    def bam_update(self, samples, vs, mu0, S0, reg):
        """
        Returns updated mean and covariance matrix with batch and match updates.
        For a batch, this is simply the mean of updates for individual samples.

        Inputs:
        samples: Array of samples of shape BxD where B is the batch dimension
        vs : Array of score functions of shape BxD corresponding to samples
        mu0 : Array of shape D, current estimate of the mean
        S0 : Array of shape DxD, current estimate of the covariance matrix

        Returns:
        mu : Array of shape D, new estimate of the mean
        S : Array of shape DxD, new estimate of the covariance matrix
        """

        assert len(samples.shape) == 2
        assert len(vs.shape) == 2
        B = samples.shape[0]
        xbar = jnp.mean(samples, axis=0)
        # jax.debug.print("Shape of samples: {samples_shape}", samples_shape=samples.shape)
        # jax.debug.print("Shape of xbar: {xbar_shape}", xbar_shape=xbar.shape)

        outer_map = jax.vmap(jnp.outer, in_axes=(0, 0))
        xdiff = samples - xbar
        C = jnp.mean(outer_map(xdiff, xdiff), axis=0)

        gbar = jnp.mean(vs, axis=0)
        gdiff = vs - gbar
        G = jnp.mean(outer_map(gdiff, gdiff), axis=0)

        U = reg * G + (reg)/(1+reg) * jnp.outer(gbar, gbar)
        V = S0 + reg * C + (reg)/(1+reg) * jnp.outer(mu0 - xbar, mu0 - xbar)
        I = jnp.identity(samples.shape[1])

        mat = I + 4 * jnp.matmul(U, V)
        # S = 2 * jnp.matmul(V, jnp.linalg.inv(I + sqrtm(mat).real))
        S = 2 * jnp.linalg.solve(I + self.get_sqrt(mat).T, V.T)

        mu = 1/(1+reg) * mu0 + reg/(1+reg) * (jnp.matmul(S, gbar) + xbar)

        return mu, S


    def bam_lowrank_update(self, samples, vs, mu0, S0, reg):
        """
        Returns updated mean and covariance matrix with low-rank BaM updates.
        For a batch, this is simply the mean of updates for individual samples.

        Inputs:
        samples: Array of samples of shape BxD where B is the batch dimension
        vs : Array of score functions of shape BxD corresponding to samples
        mu0 : Array of shape D, current estimate of the mean
        S0 : Array of shape DxD, current estimate of the covariance matrix

        Returns:
        mu : Array of shape D, new estimate of the mean
        S : Array of shape DxD, new estimate of the covariance matrix
        """

        assert len(samples.shape) == 2
        assert len(vs.shape) == 2
        B = samples.shape[0]
        xbar = jnp.mean(samples, axis=0)
        outer_map = jax.vmap(jnp.outer, in_axes=(0, 0))
        xdiff = samples - xbar
        C = jnp.mean(outer_map(xdiff, xdiff), axis=0)

        gbar = jnp.mean(vs, axis=0)
        gdiff = vs - gbar
        G = jnp.mean(outer_map(gdiff, gdiff), axis=0)

        U = reg * G + (reg)/(1+reg) * jnp.outer(gbar, gbar)
        V = S0 + reg * C + (reg)/(1+reg) * jnp.outer(mu0 - xbar, mu0 - xbar)

        # Form decomposition that is D x K
        Q = self.compute_Q((U, B))
        I = jnp.identity(B)
        VT = V.T
        A = VT.dot(Q)
        BB = 0.5*I + jnp.real(self.get_sqrt(A.T.dot(Q) + 0.25*I))
        BB = BB.dot(BB)
        CC = jnp.linalg.solve(BB, A.T)
        S = VT - A @ CC
        mu = 1/(1+reg) * mu0 + reg/(1+reg) * (jnp.matmul(S, gbar) + xbar)

        return mu, S



    def fit(self, key, regf, mean=None, cov=None, batch_size=2, niter=5000, nprint=10, verbose=True, check_goodness=True, monitor=None, retries=10, jitter=1e-6, debug_covariance=False):
        """
        Main function to fit a multivariate Gaussian distribution to the target

        Inputs:
          key: Random number generator key (jax.random.PRNGKey)
          mean : Function to return regularizer value at an iteration. See Regularizers class below
          mean : Optional, initial value of the mean. Expected None or array of size D
          cov : Optional, initial value of the covariance matrix. Expected None or array of size DxD
          batch_size : Optional, int. Number of samples to match scores for at every iteration
          niter : Optional, int. Total number of iterations
          nprint : Optional, int. Number of iterations after which to print logs
          verbose : Optional, bool. If true, print number of iterations after nprint
          check_goodness : Optional, bool. Recommended. Wether to check floating point errors in covariance matrix update
          monitor : Optional. Function to monitor the progress and track different statistics for diagnostics.
                    Function call should take the input tuple (iteration number, [mean, cov], lp, key, number of grad evals).
                    Example of monitor class is provided in utils/monitors.py

        Returns:
          mu : Array of shape D, fit of the mean
          cov : Array of shape DxD, fit of the covariance matrix
        """
        # if jitter != 0:
        #     print("Using jitter:", jitter)
        # else:
        #     print("Not using jitter")

        if mean is None:
            mean = jnp.zeros(self.D)
        if cov is None:
            cov = jnp.identity(self.D)

        nevals = 1

        if self.use_lowrank:
            update_function = self.bam_lowrank_update
        else:
            update_function = self.bam_update
        if self.jit_compile:
            update_function = jit(update_function)

        if nprint > niter: nprint = niter
        for i in range(niter + 1):
            if (i%(niter//nprint) == 0) and verbose :
                print(f'Iteration {i} of {niter}')

            if monitor is not None:
                if (i%monitor.checkpoint) == 0:
                    monitor(i, [mean, cov], self.lp, key, nevals=nevals)
                    nevals = 0

            # Can generate samples from jax distribution (commented below), but using numpy is faster
            j = 0
            while True:         # Sometimes run crashes due to a bad sample. Avoid that by re-trying.
                try:
                    key, key_sample = random.split(key, 2)
                    np.random.seed(key_sample[0])
                    # Generate samples using NumPy outside the jit-compiled function
                    samples = np.random.multivariate_normal(mean=np.array(mean), cov=np.array(cov), size=batch_size)
                    vs = self.lp_g(samples)
                    nevals += batch_size
                    reg = regf(i)
                    # Pass NumPy arrays to the jit-compiled function
                    mean_new, cov_new = update_function(jnp.array(samples), jnp.array(vs), mean, cov, reg)
                    cov_new += np.eye(self.D) * jitter # jitter covariance matrix
                    cov_new = (cov_new + cov_new.T)/2.
                    
                    break
                except Exception as e:
                    if j < retries :
                        j += 1
                        print(f"Failed with exception {e}")
                        print(f"Trying again {j} of {retries}")
                    else : raise e

            is_good = self._check_goodness(cov_new)
            if is_good:
                mean, cov = mean_new, cov_new
            else:
                if verbose: print("Bad update for covariance matrix. Revert")

        if monitor is not None:
            monitor(i, [mean, cov], self.lp, key, nevals=nevals)
        return mean, cov


    def _check_goodness(self, cov):
        '''
        Internal function to check if the new covariance matrix is a valid covariance matrix.
        Required due to floating point errors in updating the convariance matrix directly,
        insteead of it's Cholesky form.
        '''
        is_good = False
        try:
            if (np.isnan(np.linalg.cholesky(cov))).any():
                pass
            else:
                is_good = True
            return is_good
        except:
            return is_good



In [11]:
class Regularizers:
    """
    Class for regularizers used in BaM
    """

    def __init__(self):
        self.counter = 0  # 被调用次数

    def reset(self):
        self.counter = 0

    def constant(self, reg0):
        def reg_iter(iteration):
            self.counter += 1
            return reg0  # 返回常数正则化值

        return reg_iter

    def linear(self, reg0):
        def reg_iter(iteration):
            self.counter += 1
            return reg0 / self.counter  # 返回线性递减正则化值

        return reg_iter

    def custom(self, func):
        def reg_iter(iteration):
            self.counter += 1
            return func(self.counter)  # 返回自定义正则化函数

        return reg_iter

In [12]:
import jax
import jax.numpy as jnp
from jax import jit, random
from jax.scipy.linalg import sqrtm as sqrtm_jsp
from scipy.linalg import sqrtm as sqrtm_sp
import numpy as np

# import scipy.sparse as spys
from scipy.sparse.linalg import svds
from jax.lib import xla_bridge



class BaM_V2:
    """
    Wrapper class for using BaM updates to fit a distribution
    """

    def __init__(self, D, lp, lp_g, use_lowrank=False, jit_compile=True):
        """
        Inputs:
          D: (int) Dimensionality (number) of parameters
          lp : Function to evaluate target log-probability distribution.
                         (Only used in monitor, not for fitting)
          lp_g : Function to evaluate score, i.e. the gradient of the target log-probability distribution

          是否使用低秩近似（减少计算成本和内存使用）和 JIT 编译（加速计算）
        """
        #print("Using Bam v2")
        self.D = D
        self.lp = lp
        self.lp_g = lp_g
        self.use_lowrank = use_lowrank
        if use_lowrank:
            print("Using lowrank update")
        self.jit_compile = jit_compile
        if not jit_compile:
            print("Not using jit compilation. This may take longer than it needs to.")

    # 用于低秩更新
    def compute_Q_host(U_B):
        U, B = U_B
        # UU, DD, VV = spys.linalg.svds(U, k=B)
        U = np.array(U)
        UU, DD, VV = svds(U, k=B)  # 奇异值分解→三个矩阵
        # B：要计算的奇异值的个数，设为batchsize
        return UU * np.sqrt(DD)


    def compute_Q(self, U_B):
        result_shape = jax.ShapeDtypeStruct((U_B[0].shape[0], U_B[1]), U_B[0].dtype)
        return jax.pure_callback(self.compute_Q_host, result_shape, U_B)
        # 将 compute_Q_host 函数包装为一个可以在 JAX 计算图中使用的纯函数


    # 计算矩阵 M 的平方根
    def get_sqrt(self, M):
        if xla_bridge.get_backend().platform == "gpu":
            result_shape = jax.ShapeDtypeStruct(M.shape, M.dtype)
            M_root = jax.pure_callback(
                lambda x: sqrtm_sp(x).astype(M.dtype), result_shape, M
            )
            # sqrt can be complex sometimes, we only want real part
        elif xla_bridge.get_backend().platform == "cpu":
            M_root = sqrtm_jsp(M)
        else:
            print(
                "Backend not recongnized in get_sqrt function. Should be either gpu or cpu"
            )
            raise
        return M_root.real


    def check_spd(self, M, name, eps=1e-8):
        """
        用 jax.debug.print 打印 M 的对称误差、最小/最大特征值，
        以及一个 SPD? 标志（min_eig >= -eps）。
        这个函数里不做 Python 层面的 if-branch，JAX 下不会报 TracerBoolConversionError。
        """
        # 1) 对称误差
        sym_err = jnp.linalg.norm(M - M.T)

        # 2) 强制取真正对称矩阵去算特征值
        M_sym = (M + M.T) * 0.5
        w = jnp.linalg.eigvalsh(M_sym)

        # 3) SPD 判定（仍是 JAX 布尔标量）
        is_spd = w[0] >= -eps

        # 4) 打印
        jax.debug.print(
            "[SPD] {}  sym_err={:.2e}  eig_min={:.2e}  eig_max={:.2e}  SPD?={}",
            name,
            sym_err,
            w[0],
            w[-1],
            is_spd,
        )

        return M  # 返回原矩阵，不改计算图


    # 更新均值和协方差矩阵
    def bam_update(self, samples, vs, mu0, S0, reg):
        """
        Returns updated mean and covariance matrix with batch and match updates.
        For a batch, this is simply the mean of updates for individual samples.

        Inputs:
        samples: Array of samples of shape BxD where B is the batch dimension
        vs : Array of score functions of shape BxD corresponding to samples
        mu0 : Array of shape D, current estimate of the mean
        S0 : Array of shape DxD, current estimate of the covariance matrix
        reg : Regularization parameter (lambda_t)

        Returns:
        mu : Array of shape D, new estimate of the mean
        S : Array of shape DxD, new estimate of the covariance matrix
        """
        # (B, D)
        assert len(samples.shape) == 2
        assert len(vs.shape) == 2
        B = samples.shape[0]
        outer_map = jax.vmap(jnp.outer, in_axes=(0, 0))
        # 使用 jax.vmap 对 jnp.outer 进行向量化 → 可以对多个向量进行外积运算

        gbar = jnp.mean(vs, axis=0)
        zbar = jnp.mean(samples, axis=0)

        A = samples - (1 / (1 + reg)) * mu0 - (reg / (1 + reg)) * zbar
        U = reg * (
            jnp.mean(outer_map(vs, vs), axis=0) - (reg / (1 + reg)) * jnp.outer(gbar, gbar)
        )
        V = (
            reg * jnp.mean(outer_map(A, A), axis=0)
            + S0
            + (reg**2 / (1 + reg) ** 2) * jnp.outer(mu0 - zbar, mu0 - zbar)
        )

        I = jnp.identity(samples.shape[1])
        mat = I + 4 * jnp.matmul(U, V)
        S = 2 * jnp.matmul(V, jnp.linalg.inv(I + self.get_sqrt(mat)))

        mu = (
            (1 / (1 + reg)) * mu0
            + (reg / (1 + reg)) * zbar
            + (reg / (1 + reg)) * jnp.matmul(S, gbar)
        )

        return mu, S


    # 更新均值和协方差矩阵
    def bam_update_debug(self, samples, vs, mu0, S0, reg):
        """
        Returns updated mean and covariance matrix with batch and match updates.
        For a batch, this is simply the mean of updates for individual samples.

        Inputs:
        samples: Array of samples of shape BxD where B is the batch dimension
        vs : Array of score functions of shape BxD corresponding to samples
        mu0 : Array of shape D, current estimate of the mean
        S0 : Array of shape DxD, current estimate of the covariance matrix
        reg : Regularization parameter (lambda_t)

        Returns:
        mu : Array of shape D, new estimate of the mean
        S : Array of shape DxD, new estimate of the covariance matrix
        """
        # (B, D)
        assert len(samples.shape) == 2
        assert len(vs.shape) == 2

        B = samples.shape[0]
        outer_map = jax.vmap(jnp.outer, in_axes=(0, 0))
        # 使用 jax.vmap 对 jnp.outer 进行向量化 → 可以对多个向量进行外积运算

        gbar = jnp.mean(vs, axis=0)
        zbar = jnp.mean(samples, axis=0)

        A = samples - (1 / (1 + reg)) * mu0 - (reg / (1 + reg)) * zbar
        U = reg * (
            jnp.mean(outer_map(vs, vs), axis=0) - (reg / (1 + reg)) * jnp.outer(gbar, gbar)
        )
        V = (
            reg * jnp.mean(outer_map(A, A), axis=0)
            + S0
            + (reg**2 / (1 + reg) ** 2) * jnp.outer(mu0 - zbar, mu0 - zbar)
        )

        I = jnp.identity(samples.shape[1])
        mat = I + 4 * jnp.matmul(U, V)
        S = 2 * jnp.matmul(V, jnp.linalg.inv(I + get_sqrt(mat)))

        mu = (
            (1 / (1 + reg)) * mu0
            + (reg / (1 + reg)) * zbar
            + (reg / (1 + reg)) * jnp.matmul(S, gbar)
        )

        jax.debug.print("score: {}", vs)
        # ---- SPD 检查点 1: U, V, S0(旧) ----
        self.check_spd(U, "U (score-cov term)")
        self.check_spd(V, "V (batch-stats term)")
        self.check_spd(S0, "Sigma_old")

        # ---- SPD 检查点 2: S_cand(新) ----
        self.check_spd(S, "Sigma_new")

        return mu, S


    # 低秩BaM更新均值和协方差矩阵
    def bam_lowrank_update(self, samples, vs, mu0, S0, reg):
        """
        Returns updated mean and covariance matrix with low-rank BaM updates.
        For a batch, this is simply the mean of updates for individual samples.

        Inputs:
        samples: Array of samples of shape BxD where B is the batch dimension
        vs : Array of score functions of shape BxD corresponding to samples
        mu0 : Array of shape D, current estimate of the mean
        S0 : Array of shape DxD, current estimate of the covariance matrix

        Returns:
        mu : Array of shape D, new estimate of the mean
        S : Array of shape DxD, new estimate of the covariance matrix
        """

        assert len(samples.shape) == 2
        assert len(vs.shape) == 2

        B = samples.shape[0]
        xbar = jnp.mean(samples, axis=0)
        outer_map = jax.vmap(jnp.outer, in_axes=(0, 0))
        xdiff = samples - xbar
        C = jnp.mean(outer_map(xdiff, xdiff), axis=0)

        gbar = jnp.mean(vs, axis=0)
        gdiff = vs - gbar
        G = jnp.mean(outer_map(gdiff, gdiff), axis=0)

        U = reg * G + (reg) / (1 + reg) * jnp.outer(gbar, gbar)
        V = S0 + reg * C + (reg) / (1 + reg) * jnp.outer(mu0 - xbar, mu0 - xbar)

        # Form decomposition that is D x K
        # 低秩分解
        Q = self.compute_Q((U, B))  # 低秩矩阵Q，用于近似矩阵U
        I = jnp.identity(B)  # 大小为B的单位矩阵
        VT = V.T  # 转置
        A = VT.dot(Q)
        BB = 0.5 * I + jnp.real(self.get_sqrt(A.T.dot(Q) + 0.25 * I))
        BB = BB.dot(BB)
        CC = jnp.linalg.solve(BB, A.T)
        S = VT - A @ CC
        mu = 1 / (1 + reg) * mu0 + reg / (1 + reg) * (jnp.matmul(S, gbar) + xbar)

        return mu, S


    # 协方差矩阵检查函数
    def check_covariance_matrix(cov, iteration, debug=True):
        """检查协方差矩阵的性质并打印调试信息"""
        issues = []
        
        # 检查对称性
        if not np.allclose(cov, cov.T, atol=1e-10):
            issues.append("not symmetric")
        
        # 检查特征值（正定性）
        try:
            eigenvals = np.linalg.eigvals(cov)
            min_eigenval = np.min(eigenvals)
            if min_eigenval <= 0:
                issues.append(f"not positive definite (min eigenvalue: {min_eigenval:.2e})")
        except Exception as e:
            issues.append(f"eigenvalue computation failed: {e}")
        
        # 检查是否包含NaN或Inf
        if np.any(np.isnan(cov)):
            issues.append("contains NaN")
        if np.any(np.isinf(cov)):
            issues.append("contains Inf")
        
        # 条件数检查
        try:
            cond_num = np.linalg.cond(cov)
            if cond_num > 1e12:
                issues.append(f"ill-conditioned (condition number: {cond_num:.2e})")
        except Exception as e:
            issues.append(f"condition number computation failed: {e}")
        
        if debug and issues:
            print(f"[Iteration {iteration}] Covariance matrix issues: {', '.join(issues)}")
            print(f"  Matrix shape: {cov.shape}")
            print(f"  Matrix norm: {np.linalg.norm(cov):.2e}")
            try:
                print(f"  Diagonal elements range: [{np.min(np.diag(cov)):.2e}, {np.max(np.diag(cov)):.2e}]")
            except:
                pass
        
        return len(issues) == 0, issues

    def fit(
        self,
        key,
        regf,
        mean=None,
        cov=None,
        batch_size=2,
        niter=5000,
        nprint=10,
        verbose=True,
        check_goodness=True,
        monitor=None,
        retries=10,
        jitter=1e-6,
        debug_covariance=False,  # 新增调试参数
    ):
        """
        Main function to fit a multivariate Gaussian distribution to the target

        Inputs:
          key: Random number generator key (jax.random.PRNGKey)
          regf : Function to return regularizer value at an iteration. See Regularizers class below
          正则化函数
          mean : Optional, initial value of the mean. Expected None or array of size D
          cov : Optional, initial value of the covariance matrix. Expected None or array of size DxD
          batch_size : Optional, int. Number of samples to match scores for at every iteration
          niter : Optional, int. Total number of iterations
          最大迭代次数
          nprint : Optional, int. Number of iterations after which to print logs
          verbose : Optional, bool. If true, print number of iterations after nprint
          debug_covariance : bool. If true, check and print covariance matrix issues
          monitor : Optional. Function to monitor the progress and track different statistics for diagnostics.
                    Function call should take the input tuple (iteration number, [mean, cov], lp, key, number of grad evals).
                    Example of monitor class is provided in utils/monitors.py

        Returns:
          mu : Array of shape D, fit of the mean
          cov : Array of shape DxD, fit of the covariance matrix
        """

        # if jitter != 0:
        #     print("Using jitter:", jitter)
        # else:
        #     print("Not using jitter")
            
        # 初始化
        if mean is None:
            mean = jnp.zeros(self.D)
        if cov is None:
            cov = jnp.identity(self.D)  # DxD

        nevals = 1  # 梯度评估次数

        if self.use_lowrank:
            update_function = self.bam_lowrank_update
        else:
            update_function = self.bam_update
        if self.jit_compile:
            update_function = jit(update_function)

        if nprint > niter:
            nprint = niter

        for i in range(niter + 1):
            if (i % (niter // nprint) == 0) and verbose:
                print(f"Iteration {i} of {niter}")

            # 调用监控函数 monitor，并将梯度评估次数 nevals 重置为 0
            if monitor is not None:
                if (i % monitor.checkpoint) == 0:
                    monitor(i, [mean, cov], self.lp, key, nevals=nevals)
                    nevals = 0

            # Can generate samples from jax distribution (commented below), but using numpy is faster
            j = 0
            while (
                True
            ):  # Sometimes run crashes due to a bad sample. Avoid that by re-trying.
                try:
                    key, key_sample = random.split(key, 2)
                    np.random.seed(key_sample[0])

                    # 从当前高斯分布中抽取 batch_size 个样本

                    try:
                        samples = np.random.multivariate_normal(
                            mean=mean, cov=cov, size=batch_size
                        )

                        if debug_covariance:
                            print(f"[Iteration {i}] 生成 {batch_size} 个样本...")
                            print(f"当前 q 的均值: {mean}, \n 协方差矩阵:{cov}")
                            print(f"[Iteration {i}] 样本生成成功")
                            print("样本均值:", samples.mean(axis=0))  # 每个维度的平均值
                            print("样本协方差:", np.cov(samples.T))  # 各维度之间的协方差

                    except Exception as sample_error:
                        print(f"[Iteration {i}] ❌ 样本生成失败: {sample_error}")
                        print(f"  协方差矩阵统计:")
                        print(f"    形状: {np.array(cov).shape}")
                        print(f"    范围: [{np.min(cov):.2e}, {np.max(cov):.2e}]")
                        print(f"    对角线元素: {np.diag(cov)}")
                        raise sample_error

                    # 计算样本的得分
                    vs = self.lp_g(samples)
                    nevals += batch_size
                    # 正则化
                    reg = regf(i)

                    # 更新参数
                    if debug_covariance:
                        mean_new, cov_new = self.bam_update_debug(samples, vs, mean, cov, reg)
                    else:
                        mean_new, cov_new = self.bam_update(samples, vs, mean, cov, reg)

                    # 检查更新后的协方差矩阵
                    if debug_covariance:
                        print(f"[Iteration {i}] 的lambda (reg={reg:.2e})...")
                        is_valid_new, issues_new = self.check_covariance_matrix(np.array(cov_new), i, debug=False)
                        if not is_valid_new:
                            print(f"[Iteration {i}] ⚠️  更新后的协方差矩阵有问题: {', '.join(issues_new)}")
                            print(cov_new)
                            print(f"mu_{i} = {mean_new}")

                    cov_new += np.eye(self.D) * jitter # jitter covariance matrix
                    cov_new = (cov_new + cov_new.T)/2.

                    mean, cov = mean_new, cov_new

                    break

                except Exception as e:
                    if j < retries:
                        j += 1
                        print(f"[Iteration {i}] Failed with exception {e}")
                        print(f"  Trying again {j} of {retries}")
                    else:
                        print(f"[Iteration {i}] 达到最大重试次数，抛出异常")
                        raise e

        if monitor is not None:
            monitor(i, [mean, cov], self.lp, key, nevals=nevals)

        return mean, cov


## score fn（medvae 版：展平 z ↔ 空间 latent 的适配）

In [13]:
def make_lp_and_score_fn(x_target, model, sigma2=1.0):
    """
    medvae 版：BaM 侧 z 是展平的 (D,) = EMBED_DIM*LATENT_RES^2 向量，
    view 回 (B, EMBED_DIM, LATENT_RES, LATENT_RES) 后走 model.decode（post_quant_conv + decoder）。
    ⚠️ flatten/view 顺序必须一致（行优先 C,H,W），错了会静默算错 score（FID 全错不报错）。
    lp = log N(z; 0, I) + log N(x_t; dec(z), σ² I)，x 在 [-1,1] 空间（与训练 sd 的 σ²=1.0 口径一致）。
    """
    x_t = x_target.detach().to(device).unsqueeze(0)  # (1,C,H,W)

    def compute_logp(z_flat):
        lpz = -0.5 * (z_flat ** 2).sum(dim=1)
        z = z_flat.view(-1, EMBED_DIM, LATENT_RES, LATENT_RES)
        xrec = model.decode(z)
        err = x_t - xrec
        lpx = -0.5 / sigma2 * err.reshape(z_flat.size(0), -1).pow(2).sum(dim=1)
        return lpz + lpx

    def lp_fn(samples_np):
        z = torch.from_numpy(np.asarray(samples_np)).float().to(device)
        with torch.no_grad():
            logp = compute_logp(z)
        return logp.detach().cpu().numpy()

    def score_fn(samples_np):
        z = torch.from_numpy(np.asarray(samples_np)).float().to(device)
        z.requires_grad_(True)
        logp = compute_logp(z).sum()
        (grad_z,) = torch.autograd.grad(logp, z)
        return grad_z.detach().cpu().numpy()

    return lp_fn, score_fn


In [14]:
def make_bam(version="v2", **kwargs):
    if version == "v0":
        return BaM_V0(**kwargs)
    elif version == "v2":
        return BaM_V2(**kwargs)
    else:
        raise ValueError("version must be 'v0' or 'v2'")

## refinement runner

In [15]:
class TrajMonitor:
    """记录每步 mean 解码 SSE（[0,1] 显示标度，(x+1)/2 不 clamp，与 MLP 版口径可比）与 mean 轨迹。
    mse[0] = encoder 后验均值的初始重构（未 refinement）。"""

    def __init__(self, model, x_target, checkpoint=1):
        self.mse = []
        self.mus = []
        self.model = model
        self.x_target = x_target.detach().to(device).unsqueeze(0)
        self.checkpoint = checkpoint

    def __call__(self, i, params, lp, key, nevals=1):
        mu, cov = params
        mu_np = np.asarray(mu, dtype=np.float64)
        with torch.no_grad():
            z = torch.from_numpy(mu_np).float().view(1, EMBED_DIM, LATENT_RES, LATENT_RES).to(device)
            x_rec = self.model.decode(z)
        self.mse.append(torch.sum(((x_rec - self.x_target) / 2) ** 2).item())
        self.mus.append(mu_np.astype(np.float32))
        return key


def run_refine_for_image(x_tensor, verbose=False):
    """从 encoder 后验 (mu, diag(exp(logvar))) 初始化，跑 T 步 BaM refinement，返回轨迹 monitor。
    verbose=True 时 BaM 每 ~T/10 步打一行 Iteration 进度（preflight 用，正式 run 保持 False 免刷屏）。"""
    with torch.no_grad():
        posterior = model.encode(x_tensor.unsqueeze(0).to(device))
        mean_init = posterior.mean.flatten().cpu().numpy().astype(np.float64)
        cov_init = np.diag(np.exp(posterior.logvar.flatten().cpu().numpy().astype(np.float64)))

    lp_fn, score_fn = make_lp_and_score_fn(x_tensor, model, sigma2=SIGMA2)

    solver = make_bam(
        version=BAM_VERSION, D=LATENT_DIM, lp=lp_fn, lp_g=score_fn,
        use_lowrank=False, jit_compile=True,
    )
    reg = Regularizers().constant(LAMBDA_T)
    monitor = TrajMonitor(model, x_tensor, checkpoint=1)

    key = jax.random.PRNGKey(SEED)
    np.random.seed(SEED)
    mu, cov = solver.fit(
        key=key, regf=reg, mean=mean_init, cov=cov_init,
        batch_size=B, niter=T, verbose=verbose, monitor=monitor, jitter=JITTER,
    )
    return mu, cov, monitor


## preflight（先跑这个：1 张图验证曲线不是病态，再跑全量）

In [ ]:
# ===== preflight：先跑 1 张，验证曲线不是病态再跑全量 =====
# 读法：
#   曲线正常下降/稳住          -> 欠定没咬人，放心跑全量；
#   从第 1 步起纹丝不动的平线   -> 更新病态走不动（wo-KL_t 死法），结果不可用，回头议 B/λ；
#   NaN / 采样报错重试到崩     -> 数值崩溃，同上。
# 每次调用 run_refine_for_image 都会重置 np.random.seed(SEED)/PRNGKey，preflight 不影响正式 run 的可复现性。
_t0 = time.time()
_x0 = test_dataset[0]
_, _, _mon = run_refine_for_image(_x0, verbose=True)   # 每 ~10 步打一行 Iteration，确认在动
_el = time.time() - _t0
_mse = np.asarray(_mon.mse)

print(f"1 张耗时 {_el:.1f} s（~{_el/(T+1):.2f} s/step）")
print(f"全量预估：N=100 不分片单本约 {_el*100/3600:.1f} h")
print(f"SSE: init {_mse[0]:.4f} -> final {_mse[-1]:.4f}  (delta {_mse[-1]-_mse[0]:+.4f})")

assert np.all(np.isfinite(_mse)), "❌ 轨迹出现 NaN/Inf：数值崩溃，不要跑全量"
_rel_move = np.abs(_mse[1:] - _mse[0]).max() / max(abs(_mse[0]), 1e-12)
if _rel_move < 1e-6:
    print("⚠️ 曲线从第 1 步起纹丝不动（相对变化 < 1e-6）：疑似欠定卡死，结果不可用，先回头议 B/λ")
else:
    print(f"✓ 曲线有正常移动（最大相对变化 {_rel_move:.2e}）")

plt.figure(figsize=(5.2, 3.4), dpi=120)
plt.plot(_mse, lw=1.4)
plt.xlabel("step")
plt.ylabel("SSE (0-1 scale)")
plt.title(f"preflight idx=0  {DATASET}/{METHOD_TAG}  B={B}")
plt.grid(alpha=0.25, ls=":")
plt.tight_layout()
plt.show()


Iteration 0 of 100


/tmp/ipykernel_2556942/2420542429.py:58: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  if xla_bridge.get_backend().platform == "gpu":
/tmp/ipykernel_2556942/2908030654.py:16: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:203.)
  z = torch.from_numpy(mu_np).float().view(1, EMBED_DIM, LATENT_RES, LATENT_RES).to(device)


Iteration 10 of 100
Iteration 20 of 100
Iteration 30 of 100
Iteration 40 of 100
Iteration 50 of 100
Iteration 60 of 100
Iteration 70 of 100
Iteration 80 of 100
Iteration 90 of 100
Iteration 100 of 100


## λ preflight 小格子（当上面 preflight 报非 PSD 警告 / 平线时跑；选定 λ 后回 config 改 LAMBDA_T 再全量）

In [ ]:
# ===== λ 小格子：1 张图 × λ∈{10,100,1500} × 30 步，找数值健康的 λ =====
# 机制：mat = I + 4UV 的条件数 ~ (λ·score尺度)²，D=1024 下 λ=7500 超出浮点精度，
#       更新出的 Σ 带负特征值（numpy 报 covariance is not symmetric positive-semidefinite）。
#       降 λ 直接压条件数；且均值更新里旧均值权重仅 1/(1+λ)，λ=10 时每步仍走 ~91%，探针不钝
#       （mnist B=2/λ=10 从冷启动 T=100 可收敛）。KL_t 的正则主要作用在协方差更新，恰好治此病。
# 选法：非 PSD 警告 = 0 且曲线下降/稳住的【最大】λ（λ 越大越接近原协议的「弱 proximal」口径）。
import warnings as _warnings

LAMBDA_GRID = [10, 100, 1500]
T_GRID = 20          # 格子只跑 20 步看数值健康度（判断非 PSD/走向足够）；选定 λ 后全量仍用 T=100
_x0 = test_dataset[0]

_results = {}
for _lam in LAMBDA_GRID:
    _lp, _score = make_lp_and_score_fn(_x0, model, sigma2=SIGMA2)
    with torch.no_grad():
        _post = model.encode(_x0.unsqueeze(0).to(device))
        _mean0 = _post.mean.flatten().cpu().numpy().astype(np.float64)
        _cov0 = np.diag(np.exp(_post.logvar.flatten().cpu().numpy().astype(np.float64)))
    _solver = make_bam(version=BAM_VERSION, D=LATENT_DIM, lp=_lp, lp_g=_score,
                       use_lowrank=False, jit_compile=True)
    _mon = TrajMonitor(model, _x0, checkpoint=1)
    _t0 = time.time()
    with _warnings.catch_warnings(record=True) as _wlist:
        _warnings.simplefilter("always")
        _solver.fit(key=jax.random.PRNGKey(SEED), regf=Regularizers().constant(_lam),
                    mean=_mean0, cov=_cov0, batch_size=B, niter=T_GRID,
                    verbose=True, monitor=_mon, jitter=JITTER)   # 每 2 步打一行 Iteration 进度
    _npsd = sum(1 for w in _wlist if "positive-semidefinite" in str(w.message))
    _mse = np.asarray(_mon.mse)
    _el = time.time() - _t0
    _results[_lam] = (_mse, _npsd, _el)
    print(f"λ={_lam:>6}: 非PSD警告 {_npsd}/{T_GRID+1} 步  "
          f"SSE {_mse[0]:.2f} -> {_mse[-1]:.2f}  finite={np.all(np.isfinite(_mse))}  {_el:.0f}s",
          flush=True)

plt.figure(figsize=(6.2, 3.8), dpi=120)
for _lam, (_mse, _npsd, _) in _results.items():
    plt.plot(_mse, lw=1.4, label=f"λ={_lam} (非PSD {_npsd})")
plt.xlabel("step")
plt.ylabel("SSE (0-1 scale)")
plt.title(f"λ preflight  {DATASET}/{METHOD_TAG}  B={B}  T={T_GRID}")
plt.legend()
plt.grid(alpha=0.25, ls=":")
plt.tight_layout()
plt.show()
print("选：非PSD=0 且曲线下降/稳住的最大 λ；回 config 改 LAMBDA_T 后再跑全量（vae/sdvae 两本用同一 λ）")


## run（固定种子抽样 + 分片；先看第 1 张的 s/img 再决定规模）

In [ ]:
N_TEST = len(test_dataset)
rng = np.random.default_rng(SEED)
if N_IMAGES is None or N_IMAGES >= N_TEST:
    chosen = np.arange(N_TEST)
else:
    # vae/sdvae 两本同 SEED 同 N_IMAGES -> 抽样完全一致，保证逐图配对
    chosen = np.sort(rng.choice(N_TEST, size=N_IMAGES, replace=False))
shard_idx = chosen[np.arange(len(chosen)) % NUM_SHARDS == SHARD_ID]   # 交错分片，难度均衡
print(f"抽样 {len(chosen)}/{N_TEST} 张; shard {SHARD_ID}/{NUM_SHARDS}: {len(shard_idx)} images")

mse_curves, mu_curves = [], []
t0 = time.time()
for n, idx in enumerate(shard_idx):
    x = test_dataset[int(idx)]          # (C,H,W) in [-1,1]
    _, _, mon = run_refine_for_image(x)
    mse_curves.append(np.asarray(mon.mse))
    mu_curves.append(np.stack(mon.mus))
    el = time.time() - t0
    print(f"{n+1}/{len(shard_idx)}  idx={int(idx)}  {el/(n+1):.1f} s/img  "
          f"ETA {(len(shard_idx)-n-1)*el/(n+1)/3600:.2f} h", flush=True)

out = f"{SAVE_DIR}/{DATASET}_medvae_{METHOD_TAG}_T{T}_B{B}_{N_TAG}_shard{SHARD_ID}of{NUM_SHARDS}.npz"
np.savez_compressed(
    out,
    indices=shard_idx,
    mse_curves=np.stack(mse_curves),                    # [n, T+2] 列0=初始，列-1=第T步后
    mu_curves=np.stack(mu_curves).astype(np.float32),   # [n, T+2, D]
    T=T, B=B, lambda_t=LAMBDA_T, sigma2=SIGMA2, version=BAM_VERSION, ckpt=CKPT,
)
print("saved:", out)


## stats（所有分片完成后）

In [ ]:
# ===== 统计：全局配对检验（所有分片跑完后运行）=====
import glob
from scipy import stats as sps

files = sorted(glob.glob(f"{SAVE_DIR}/{DATASET}_medvae_{METHOD_TAG}_T{T}_B{B}_{N_TAG}_shard*of{NUM_SHARDS}.npz"))
print(len(files), "shards:", files)

idx_list, curve_list = [], []
for f in files:
    d = np.load(f)
    idx_list.append(d["indices"])
    curve_list.append(d["mse_curves"])
indices = np.concatenate(idx_list)
curves = np.concatenate(curve_list)
order = np.argsort(indices)
indices, curves = indices[order], curves[order]
print("N =", len(indices), " curve len =", curves.shape[1])

mse_init = curves[:, 0]     # SDVAE/VAE amortized 初始 SSE
mse_final = curves[:, -1]   # 第 T 步 refinement 后 SSE
delta = mse_final - mse_init  # <0 = refinement 有改善

print(f"init  MSE: {mse_init.mean():.4f} ± {mse_init.std():.4f}")
print(f"final MSE: {mse_final.mean():.4f} ± {mse_final.std():.4f}")
print(f"delta    : {delta.mean():.4f} ± {delta.std():.4f}")
ci = sps.t.interval(0.95, len(delta) - 1, loc=delta.mean(), scale=sps.sem(delta))
print(f"delta 95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"改善比例 (delta<0): {(delta < 0).mean():.2%}")

t_stat, p_t = sps.ttest_rel(mse_final, mse_init)
w_stat, p_w = sps.wilcoxon(mse_final, mse_init)
print(f"paired t-test: t={t_stat:.3f}, p={p_t:.3g}")
print(f"Wilcoxon signed-rank: W={w_stat:.3f}, p={p_w:.3g}")


In [ ]:
# ===== 逐图统计 + 诊断图（样式同 MLP 版）=====
# 每张图：其 100 步轨迹相对初始值的差 {MSE_t - MSE_0} 做单样本 Wilcoxon。
# 注意：单图轨迹自相关，此 p 仅作描述统计；全局结论以上一 cell 的跨图配对检验为准。
diffs = curves[:, 1:] - curves[:, [0]]   # [N, T+1]
pvals = np.empty(len(curves))
for i, dd in enumerate(diffs):
    try:
        pvals[i] = sps.wilcoxon(dd).pvalue
    except ValueError:                    # 差值全 0
        pvals[i] = 1.0

method_color = {"vae": "#4C72B0", "sdvae": "#C05A2A"}[METHOD_TAG]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=150)
bp = axes[0].boxplot(
    [delta], showfliers=False, widths=0.45, patch_artist=True,
    boxprops=dict(linewidth=1.2, edgecolor="#BDBDBD"),
    whiskerprops=dict(linewidth=1.1, color="#BDBDBD"),
    capprops=dict(linewidth=1.1, color="#BDBDBD"),
    medianprops=dict(linewidth=1.5, color="white"),
)
bp["boxes"][0].set_facecolor(method_color)
bp["boxes"][0].set_alpha(0.78)
axes[0].axhline(0, color="#FF6B6B", lw=1.2, ls="--")
axes[0].set_xticks([1])
axes[0].set_xticklabels([METHOD_TAG.upper()])
axes[0].set_ylabel(r"$\Delta$MSE after 100 BAM steps (final $-$ init)")
axes[0].yaxis.grid(True, alpha=0.25, ls=":")
axes[0].spines[["top", "right"]].set_visible(False)
mean_delta = delta.mean()
axes[0].plot(1, mean_delta, marker="D", ms=6, color="black",
             markeredgecolor="white", markeredgewidth=1.0, zorder=4)
axes[0].annotate(f"mean {mean_delta:+.2f}", xy=(1.18, mean_delta),
                 va="center", fontsize=10)

frac = (pvals < 0.05).mean()
xs = np.sort(pvals)
ys = np.arange(1, len(xs) + 1) / len(xs)
axes[1].step(xs, ys, where="post", color=method_color, lw=1.4)
axes[1].axvline(0.05, color="#FF6B6B", lw=1.1, ls="--")
axes[1].plot([0.05], [frac], "o", ms=4, color="black")
axes[1].text(0.35, 0.55, f"p<0.05: {frac:.2%}", transform=axes[1].transAxes,
             fontsize=10)
axes[1].set_xlim(-0.02, 1.0)
axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel("per-image Wilcoxon p")
axes[1].set_ylabel("cumulative fraction of images")
axes[1].set_title("per-image p-value ECDF")
axes[1].grid(True, alpha=0.2, ls=":")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/{DATASET}_medvae_{METHOD_TAG}_T{T}_diagnostic.png",
            dpi=200, bbox_inches="tight")
plt.show()
print(f"p<0.05 的图像比例: {frac:.2%}")

np.savez_compressed(
    f"{SAVE_DIR}/{DATASET}_medvae_{METHOD_TAG}_T{T}_{N_TAG}_stats.npz",
    indices=indices, mse_init=mse_init, mse_final=mse_final, delta=delta, pvals=pvals,
)


In [ ]:
# ===== VAE vs SDVAE：ΔMSE 并排箱线图（medvae 版）=====
# 直接从两种方法的 npz 加载，不依赖之前的内核状态。
# 推荐运行顺序：先跑同数据集 VAE notebook，再跑 SDVAE notebook；本 cell 两本都有，缺文件会自动跳过。
import glob
from scipy import stats as sps

DATASET_LABELS = {
    "mnist": "MNIST",
    "fashion-mnist": "Fashion-MNIST",
    "cifar10": "CIFAR-10",
    "cifar100": "CIFAR-100",
    "tiny-image-net": "Tiny-ImageNet",
}

def load_idx_delta(method_tag):
    shard_files = sorted(glob.glob(
        f"{SAVE_DIR}/{DATASET}_medvae_{method_tag}_T{T}_B{B}_{N_TAG}_shard*of{NUM_SHARDS}.npz"
    ))
    if not shard_files:
        raise FileNotFoundError(
            f"没找到 {DATASET}/medvae/{method_tag} 的 npz；请先完整运行同数据集的 {method_tag} notebook"
        )
    idx, cur = [], []
    for path in shard_files:
        data = np.load(path)
        idx.append(data["indices"])
        cur.append(data["mse_curves"])
    idx = np.concatenate(idx)
    cur = np.concatenate(cur)
    o = np.argsort(idx)
    return idx[o], cur[o][:, -1] - cur[o][:, 0]   # ΔMSE = final - init，<0 = refinement 改善

try:
    loaded = {method: load_idx_delta(method) for method in ["vae", "sdvae"]}
except FileNotFoundError as exc:
    print(f"跳过组合箱线图：{exc}")
else:
    if not np.array_equal(loaded["vae"][0], loaded["sdvae"][0]):
        raise ValueError("VAE/SDVAE 抽样索引不一致；请确认两本 notebook 的 SEED/N_IMAGES/分片均一致且跑完")
    deltas = {m: loaded[m][1] for m in loaded}

    colors = ["#4C72B0", "#C05A2A"]
    fig, ax = plt.subplots(figsize=(6.8, 4.8), dpi=150)
    bp = ax.boxplot(
        [deltas["vae"], deltas["sdvae"]],
        showfliers=False, widths=0.46, patch_artist=True,
        boxprops=dict(linewidth=1.25, edgecolor="#BDBDBD"),
        whiskerprops=dict(linewidth=1.15, color="#BDBDBD"),
        capprops=dict(linewidth=1.15, color="#BDBDBD"),
        medianprops=dict(linewidth=1.6, color="white"),
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.78)

    ax.axhline(0, color="#FF6B6B", lw=1.25, ls="--")
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["VAE", "SDVAE"])
    ax.set_ylabel(r"$\Delta$MSE after 100 BAM steps (final $-$ init)")
    ax.set_title(
        f"Post-amortization refinement (MedVAE backbone), "
        f"{DATASET_LABELS.get(DATASET, DATASET)} (N={len(deltas['vae'])})"
    )
    ax.yaxis.grid(True, alpha=0.25, ls=":")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(0.55, 2.75)

    for x, method in [(1, "vae"), (2, "sdvae")]:
        values = deltas[method]
        mean_value = values.mean()
        ax.plot(
            x, mean_value, marker="D", ms=6, color="black",
            markeredgecolor="white", markeredgewidth=1.0, zorder=4,
        )
        ax.annotate(
            f"mean {mean_value:+.2f}", xy=(x + 0.25, mean_value),
            va="center", fontsize=10,
        )

    plt.tight_layout()
    png_path = f"{SAVE_DIR}/{DATASET}_medvae_vae_vs_sdvae_T{T}_box.png"
    plt.savefig(png_path, dpi=220, bbox_inches="tight")

    # 论文/汇报版：去 title，导出矢量 PDF。
    ax.set_title("")
    plt.tight_layout()
    pdf_path = f"{SAVE_DIR}/{DATASET}_medvae_vae_vs_sdvae_T{T}_box.pdf"
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    print("saved:", png_path)
    print("saved:", pdf_path)

    print(f"{'':8s}{'mean ΔMSE':>12s}{'95% CI':>22s}{'改善比例':>10s}{'单侧p(有改善)':>16s}")
    for method in ["vae", "sdvae"]:
        values = deltas[method]
        lo, hi = sps.t.interval(
            0.95, len(values) - 1, loc=values.mean(), scale=sps.sem(values)
        )
        p_improve = sps.ttest_1samp(values, 0, alternative="less").pvalue
        print(
            f"{method:8s}{values.mean():>+12.4f}   [{lo:+.4f}, {hi:+.4f}]"
            f"{(values < 0).mean():>10.2%}{p_improve:>16.3g}"
        )


In [ ]:
# ===== 可选：抽样集 FID（t=0 vs t=T）=====
# FID 定义在图像集合上，单张图无法测量 FID，所以逐图分析用 MSE；
# 这里用存下的 mu 轨迹算抽样集（N_IMAGES 张）在 t=0 / t=T 的 FID。N=100 时 FID 噪声较大，仅作参考。
from torchmetrics.image.fid import FrechetInceptionDistance
if not hasattr(np, 'float_'):
    np.float_ = np.float64

mu_list = []
for f in files:
    d = np.load(f)
    mu_list.append(d["mu_curves"])
mus = np.concatenate(mu_list)[order]   # [N, T+2, D]


def fid_at(col, batch=64):
    fid = FrechetInceptionDistance(feature=2048).to(device)  # 全局单实例累积，末尾 compute 一次
    for s0 in range(0, len(indices), batch):
        sel = indices[s0:s0 + batch].astype(np.int64)
        real = torch.stack([test_dataset[int(i)] for i in sel]).to(device)
        with torch.no_grad():
            z = (torch.from_numpy(mus[s0:s0 + batch, col]).float()
                 .view(-1, EMBED_DIM, LATENT_RES, LATENT_RES).to(device))
            fake = model.decode(z)
        real01 = ((real + 1) / 2).clamp(0, 1)
        fake01 = ((fake + 1) / 2).clamp(0, 1)
        if config.IMG_CHANNELS == 1:
            real01 = real01.repeat(1, 3, 1, 1)
            fake01 = fake01.repeat(1, 3, 1, 1)
        real_rs = F.interpolate(real01, size=(299, 299), mode="bilinear", align_corners=False)
        fake_rs = F.interpolate(fake01, size=(299, 299), mode="bilinear", align_corners=False)
        fid.update((real_rs * 255).to(torch.uint8), real=True)
        fid.update((fake_rs * 255).to(torch.uint8), real=False)
    return fid.compute().item()

print("FID t=0 (amortized):", fid_at(0))
print(f"FID t={T} (refined) :", fid_at(-1))
for t in [1, 2, 5, 10, 20, 50]:
    print(f"FID t={t} :", fid_at(t))
